# Testing the muben functions this repo actually uses

`trace_al_loop.ipynb` treated the embedding backbones as a black box (fake random embeddings). This notebook does the opposite: it calls the **real `muben` functions** — no fake data, real pretrained checkpoints, real forward passes — for the two backbones that actually go through `muben` (GROVER and classic UniMol). MoLFormer is NOT muben-based (it's plain HuggingFace `transformers`), so it's out of scope here. UniMol2 is also not muben-based (separate in-repo `unimol2/` package).

This is exactly the code that runs in `compute_grover_embeddings_chunk.py` and `compute_unimol_embeddings_chunk.py` at full (99.5M-molecule) scale — just on a handful of real molecules so every step is inspectable and fast.

Run with the repo's `py310` conda environment as the kernel. First run will be slower (loading ~190MB-1.3GB checkpoint files, generating real 3D conformers via RDKit).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/N/slate/mengjing/repos/al-molecular")
MODEL_ZOO = REPO_ROOT / "models"
assert REPO_ROOT.exists() and MODEL_ZOO.exists()
sys.path.insert(0, str(REPO_ROOT))

# muben is a subdirectory symlink target, not an installed package -- same
# sys.path trick compute_grover_embeddings_chunk.py / compute_unimol_embeddings_chunk.py use.
_muben_root = REPO_ROOT / "muben"
assert _muben_root.exists()
sys.path.insert(0, str(_muben_root))

import numpy as np
import torch
np.set_printoptions(precision=3, suppress=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## Toy molecules

A handful of real, valid SMILES (real RDKit parsing / conformer generation will run on these, so keep this list small).

In [ ]:
smiles = [
    "CCO",                            # ethanol
    "c1ccccc1",                       # benzene
    "CC(=O)Oc1ccccc1C(=O)O",          # aspirin
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",   # caffeine
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",     # ibuprofen
    "Cc1ccccc1",                      # toluene
]
print(len(smiles), "molecules")

## Part A — GROVER

GROVER needs no conformers (2D graph only). This is `compute_grover_embeddings_chunk.py::compute_embeddings_for_chunk()`, copied verbatim: build a `_GroverConfig` attribute bag, construct `CollatorGrover` + the `GROVER` model (loads `models/grover/grover_base.pt`), turn each SMILES into a `MolGraphAttrs` object via `DatasetGrover.get_mol_attr()`, collate into a batch, forward through `model.grover(...)`, then pool with `model.readout()` on **both** the bond-view and atom-view representations (using `a_scope`, the batch's atom-index ranges) and concatenate them — that concat is GROVER's actual embedding.

In [ ]:
from muben.dataset.dataset_grover.dataset import DatasetGrover
from muben.dataset.dataset_grover import CollatorGrover
from muben.model import GROVER


class _GroverConfig:
    """Same attribute bag compute_grover_embeddings_chunk.py uses -- copied
    verbatim, not re-derived, so this notebook exercises the identical
    model configuration production runs with."""
    def __init__(self, checkpoint_path):
        self.checkpoint_path = str(checkpoint_path)
        self.disable_checkpoint_loading = False
        self.hidden_size = 128
        self.dropout = 0.0
        self.bias = False
        self.num_mt_block = 1
        self.num_attn_head = 4
        self.embedding_output_type = "both"
        self.ffn_num_layers = 2
        self.ffn_hidden_size = 128
        self.activation = "ReLU"
        self.uncertainty_method = "none"
        self.task_type = "regression"
        self.bbp_prior_sigma = 0.5
        self.n_lbs = 1
        self.n_tasks = 1


grover_checkpoint = MODEL_ZOO / "grover" / "grover_base.pt"
assert grover_checkpoint.exists(), grover_checkpoint

grover_config = _GroverConfig(checkpoint_path=grover_checkpoint)
grover_collator = CollatorGrover(grover_config)
grover_model = GROVER(grover_config).to(DEVICE)
grover_model.eval()
print("GROVER model loaded from", grover_checkpoint)

In [ ]:
# One MolGraphAttrs object per SMILES -- real RDKit-derived 2D graph features.
grover_items = [
    {
        "molecule_graphs": DatasetGrover.get_mol_attr(smi),
        "lbs": np.zeros(1, dtype=np.float32),
        "masks": np.ones(1, dtype=np.float32),
    }
    for smi in smiles
]

grover_batch = grover_collator(grover_items)
grover_batch.to(DEVICE)

components = grover_batch.molecule_graphs.components
_, _, _, _, _, a_scope, _, _ = components
print("a_scope (per-molecule atom index ranges):", a_scope)

with torch.no_grad():
    out = grover_model.grover(components)
    mol_from_bond = grover_model.readout(out["atom_from_bond"], a_scope)
    mol_from_atom = grover_model.readout(out["atom_from_atom"], a_scope)
    grover_embeddings = torch.cat([mol_from_bond, mol_from_atom], dim=1).float().cpu().numpy()

print("\nGROVER embeddings shape:", grover_embeddings.shape)
for s, row in zip(smiles, grover_embeddings):
    print(f"  {row[:4]} ...  {s}")

## Part B — UniMol (classic)

UniMol needs real 3D conformers first. In production this is a two-stage pipeline (`generate_unimol_conformers_chunk.py` writes conformers to LMDB shards, `compute_unimol_embeddings_chunk.py` reads them back); here we skip the LMDB round-trip and call the same conformer-generation function directly in memory, since the LMDB step is just storage, not part of what we're testing.

**Includes the conformer-duplication fix found this session**: `muben.utils.chem.smiles_to_coords(smi, n_conformer=1)` always appends an extra 2D-fallback conformer on top of the 1 requested 3D one — i.e. it returns *2* conformers, not 1. Production code (and this notebook) keeps only index 0.

In [ ]:
from muben.utils.chem import smiles_to_coords

atoms_list, coords_list = [], []
for smi in smiles:
    atoms, coordinates = smiles_to_coords(smi, n_conformer=1)
    print(f"{smi:35s}  {len(atoms)} atoms (incl. H), {len(coordinates)} conformers returned (keeping only index 0)")
    atoms_list.append(atoms)
    coords_list.append(coordinates[:1])  # the fix: index 0 only, drop the always-appended extra

Now build the pieces `compute_unimol_embeddings_chunk.py::load_chunk_dataset()` builds, but sourcing `atoms`/`coordinates` from the lists above instead of an LMDB shard: `DictionaryUniMol` (built-in vocab, no external file), `ProcessingPipeline`, a manually-populated `DatasetUniMol` (`set_processor_variant("inference")` — loops over however many conformers are actually present, vs. `"training"` which asserts exactly 11), then `CollatorUniMol` and the `UniMol` model itself (loads `models/unimol/mol_pre_all_h_220816.pt`).

In [ ]:
from muben.dataset import DatasetUniMol
from muben.dataset.dataset_unimol.dictionary import DictionaryUniMol
from muben.dataset.dataset_unimol.process import ProcessingPipeline
from muben.dataset.dataset_unimol import CollatorUniMol
from muben.model.unimol.unimol import UniMol
from torch.utils.data import DataLoader
import types


class _UniMolConfig:
    """Same attribute bag compute_unimol_embeddings_chunk.py uses -- copied verbatim."""
    def __init__(self, checkpoint_path):
        self.checkpoint_path = str(checkpoint_path)
        self.disable_checkpoint_loading = False
        self.feature_type = "unimol"
        self.task_type = "regression"
        self.uncertainty_method = "none"
        self.bbp_prior_sigma = 0.5
        self.n_lbs = 1
        self.n_tasks = 1
        self.dropout = 0.0
        self.max_atoms = 64
        self.max_seq_len = 80
        self.only_polar_hydrogens = False
        self.remove_hydrogen = True
        self.remove_polar_hydrogen = False
        self.encoder_embed_dim = 512
        self.encoder_layers = 15
        self.encoder_attention_heads = 64
        self.encoder_ffn_embed_dim = 2048
        self.activation_fn = "gelu"
        self.pooler_stride = 1
        self.pooler_dropout = 0.0
        self.emb_dropout = 0.1
        self.attention_dropout = 0.1
        self.activation_dropout = 0.0
        self.delta_pair_repr_norm_loss = -1
        self.masked_coord_loss = 0.0
        self.masked_dist_loss = 0.0
        self.masked_type_loss = 0.0
        self.pooler_activation_fn = "Tanh"


unimol_checkpoint = MODEL_ZOO / "unimol" / "mol_pre_all_h_220816.pt"
assert unimol_checkpoint.exists(), unimol_checkpoint
unimol_config = _UniMolConfig(checkpoint_path=unimol_checkpoint)

dictionary = DictionaryUniMol.load()
dictionary.add_symbol("[MASK]", is_special=True)

unimol_dataset = DatasetUniMol()
unimol_dataset._partition = "test"
unimol_dataset.processing_pipeline = ProcessingPipeline(
    dictionary=dictionary, max_atoms=unimol_config.max_atoms, max_seq_len=unimol_config.max_seq_len,
    remove_hydrogen_flag=unimol_config.remove_hydrogen, remove_polar_hydrogen_flag=unimol_config.remove_polar_hydrogen,
)
unimol_dataset.set_processor_variant("inference")

n = len(smiles)
unimol_dataset._smiles = smiles
unimol_dataset._lbs = np.zeros((n, 1), dtype=np.float32)
unimol_dataset._masks = np.ones((n, 1), dtype=np.float32)
unimol_dataset._ori_ids = None
unimol_dataset._atoms = atoms_list
unimol_dataset._coordinates = coords_list

print("dataset built:", len(unimol_dataset._atoms), "molecules")

In [ ]:
# Same seed-before-construction fix from this session: UniMol.__init__ builds
# `hidden_layer` (the final projection applied to the CLS token) AFTER
# init_bert_params, then loads the checkpoint with strict=False -- the
# checkpoint doesn't cover hidden_layer, so it's left at PyTorch's random
# init. Any fixed seed makes this reproducible run-to-run; production uses
# a specific value (20260813) so every chunk agrees -- for this isolated
# test any fixed value works, we just need determinism within the notebook.
torch.manual_seed(0)

unimol_collator = CollatorUniMol(unimol_config, dictionary)
pad_idx = dictionary.pad()
unimol_collator._atom_pad_idx = pad_idx
unimol_collator.pad_idx = pad_idx
unimol_collator.atom_pad_idx = pad_idx

unimol_loader = DataLoader(
    unimol_dataset, batch_size=8, shuffle=False, collate_fn=unimol_collator,
    num_workers=0,
)

unimol_model = UniMol(config=unimol_config, dictionary=dictionary).to(DEVICE)


def _get_embeddings(self, batch):
    """Same get_embeddings monkey-patch compute_unimol_embeddings_chunk.py
    attaches -- UniMol's forward() is written for supervised training
    (predicts a task output), this bypasses that and returns the pooled
    CLS representation after the hidden_layer projection instead."""
    src_tokens, src_distance, src_edge_type = batch.atoms, batch.distances, batch.edge_types
    padding_mask = src_tokens.eq(self.padding_idx)
    if not padding_mask.any():
        padding_mask = None
    x = self.embed_tokens(src_tokens)
    n_node = src_distance.size(-1)
    gbf_feat = self.gbf(src_distance, src_edge_type)
    gbf_result = self.gbf_proj(gbf_feat)
    attn_bias = gbf_result.permute(0, 3, 1, 2).contiguous().view(-1, n_node, n_node)
    encoder_rep, _, _, _, _ = self.encoder(x, padding_mask=padding_mask, attn_mask=attn_bias)
    return self.hidden_layer(encoder_rep[:, 0, :])


unimol_model.get_embeddings = types.MethodType(_get_embeddings, unimol_model)
unimol_model.eval()
print("UniMol model loaded from", unimol_checkpoint)

In [ ]:
unimol_parts = []
with torch.no_grad():
    for batch in unimol_loader:
        batch.to(DEVICE)
        feat = unimol_model.get_embeddings(batch)
        unimol_parts.append(feat.float().cpu().numpy())
unimol_embeddings = np.vstack(unimol_parts)

print("UniMol embeddings shape:", unimol_embeddings.shape)
for s, row in zip(smiles, unimol_embeddings):
    print(f"  {row[:4]} ...  {s}")

## Where this maps back to the real pipeline

| Notebook piece | Real file |
|---|---|
| `_GroverConfig`, GROVER forward+pool | `compute_grover_embeddings_chunk.py::compute_embeddings_for_chunk()` |
| `DatasetGrover.get_mol_attr`, `CollatorGrover`, `GROVER` | `muben/muben/dataset/dataset_grover/`, `muben/muben/model/` |
| `smiles_to_coords` (in-memory here, LMDB in production) | `generate_unimol_conformers_chunk.py`, `muben/muben/utils/chem.py` |
| `_UniMolConfig`, `load_chunk_dataset` shape, `get_embeddings` monkey-patch | `compute_unimol_embeddings_chunk.py` |
| `DictionaryUniMol`, `ProcessingPipeline`, `DatasetUniMol`, `CollatorUniMol`, `UniMol` | `muben/muben/dataset/dataset_unimol/`, `muben/muben/model/unimol/` |
| `models/grover/grover_base.pt`, `models/unimol/mol_pre_all_h_220816.pt` | symlinked to `FusionAL/models/` |

**Not covered here:**
- MoLFormer (`compute_molformer_embeddings_chunk.py`) — plain HuggingFace `transformers`, not `muben`.
- UniMol2 (`compute_unimol2_embeddings_chunk.py`, `unimol2/`) — separate in-repo package, not `muben`.
- Fine-tuning (`backbone_finetuner.py`) — uses these same `muben` classes but with gradients on and a live conformer cache; a heavier, stateful test worth its own notebook.
- The full sharded/LMDB extraction pipeline itself (chunk bounds, resumability, `_verify_alignment`) — this notebook skips straight to the model forward pass.